# Optimizing `daily_multi_window_f` with LazyFrame

The original `daily_multi_window_f` calls `.collect()` inside each loop
iteration and then joins **eager** DataFrames. This forces Polars to
materialise intermediate results, preventing query-plan optimisation.

The fix: keep everything **lazy**, join the LazyFrames, and call
`.collect()` only once at the very end.

In [ ]:
import os
from pathlib import Path
from datetime import timedelta, date

import polars as pl

import sys
_dg = Path(os.path.abspath("")).parent / "dg"
for pkg in ("exprs/src", "feng/src", "synth-data/src"):
    sys.path.insert(0, str(_dg / pkg))

from synth_data import create_daily_agg_pf
from exprs.windows import agg_in_w_exprs

In [ ]:
daily_agg_pf = create_daily_agg_pf(
    start=date(2026, 1, 1),
    end=date(2026, 4, 1),
    min_types_per_uid=2,
    max_types_per_uid=5,
)
daily_agg_pf.head()

## Helper: single-window computation (unchanged)

In [ ]:
def daily_window_f(
    daily_agg_lf: pl.LazyFrame, current_date: date, period: int
) -> pl.LazyFrame:
    period_dt = timedelta(days=period)
    return (
        daily_agg_lf.filter(
            (current_date - period_dt < pl.col("date"))
            & (pl.col("date") <= current_date)
        )
        .group_by(["uid", "val_type"])
        .agg(agg_in_w_exprs(w=f"{period}"))
    )

## Original version (eager joins inside the loop)

Each iteration calls `.collect()`, which materialises the result and
prevents Polars from fusing the query plans across windows.

In [ ]:
def daily_multi_window_f_original(
    daily_agg_lf: pl.LazyFrame, current_date: date, periods: list[int]
) -> pl.DataFrame:
    daily_f_all: pl.DataFrame = pl.DataFrame()
    for period in periods:
        if daily_f_all.is_empty():
            daily_f_all = daily_window_f(
                daily_agg_lf=daily_agg_lf, current_date=current_date, period=period
            ).collect()
        else:
            daily_f_all = daily_f_all.join(
                daily_window_f(
                    daily_agg_lf=daily_agg_lf, current_date=current_date, period=period
                ).collect(),
                on=["uid", "val_type"],
                how="full",
                coalesce=True,
            )
    return daily_f_all

## Optimized version (lazy joins, single collect)

All window LazyFrames are joined **lazily**. Polars merges the query
plans so that the source parquet is scanned only once, predicates are
pushed down, and projections are pruned across all windows.

In [ ]:
def daily_multi_window_f_optimized(
    daily_agg_lf: pl.LazyFrame, current_date: date, periods: list[int]
) -> pl.DataFrame:
    if not periods:
        return pl.DataFrame()

    frames = [
        daily_window_f(daily_agg_lf, current_date, p)
        for p in periods
    ]

    result_lf = frames[0]
    for lf in frames[1:]:
        result_lf = result_lf.join(
            lf, on=["uid", "val_type"], how="full", coalesce=True
        )

    return result_lf.collect()  # single materialisation

## Comparison

In [ ]:
current_date = date(2026, 3, 15)

original = daily_multi_window_f_original(
    daily_agg_pf.lazy(), current_date, [3, 7, 14]
).sort(["uid", "val_type"])

optimized = daily_multi_window_f_optimized(
    daily_agg_pf.lazy(), current_date, [3, 7, 14]
).sort(["uid", "val_type"])

print("Results match:", original.equals(optimized))
optimized.head(10)

## Why this matters

| Aspect | Original | Optimized |
|--------|----------|-----------|
| `.collect()` calls | N (one per period) | 1 |
| Query plan fusion | No | Yes |
| Predicate push-down across joins | No | Yes |
| Memory for intermediates | O(N) DataFrames | Single plan |

For large datasets and many window periods the optimized version can
be significantly faster because Polars optimises the full query graph
before executing anything.